In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('dataset.csv')

In [ ]:
df

Removendo valores nulos

In [ ]:
df_fixed = df.dropna()

In [ ]:
df_fixed = df_fixed.reset_index()

In [ ]:
df_fixed = df_fixed.drop('Modelo', axis=1)

In [ ]:
df['Marca'].unique()

In [ ]:
for linha in range(len(df_fixed)):
    df_fixed['Tipo da Mem?ria'].iloc[linha] = df_fixed['Tipo da Mem?ria'].iloc[linha].upper().strip()

    if df_fixed['Marca'].iloc[linha].upper().find('MSI') != -1:
        df_fixed['Marca'].iloc[linha] = 'MSI'

    elif df_fixed['Marca'].iloc[linha].upper().find('GALAX') != -1:
        df_fixed['Marca'].iloc[linha] = 'GALAX'

    elif df_fixed['Marca'].iloc[linha].upper().find('ASUS') != -1:
        df_fixed['Marca'].iloc[linha] = 'ASUS'

    elif df_fixed['Marca'].iloc[linha].upper().find('PALIT') != -1:
        df_fixed['Marca'].iloc[linha] = 'PALIT'
    
    elif df_fixed['Marca'].iloc[linha].upper().find('ZOTAC') != -1:
        df_fixed['Marca'].iloc[linha] = 'ZOTAC'

    elif df_fixed['Marca'].iloc[linha].upper().find('POWERCOLOR') != -1:
        df_fixed['Marca'].iloc[linha] = 'POWERCOLOR'

    elif df_fixed['Marca'].iloc[linha].upper().find('ASROCK') != -1:
        df_fixed['Marca'].iloc[linha] = 'ASROCK'
    
    elif df_fixed['Marca'].iloc[linha].upper().find('NEOLOGIC') != -1:
        df_fixed['Marca'].iloc[linha] = 'NEOLOGIC'

In [ ]:
df_fixed['Marca'].unique()

In [ ]:
df_fixed

In [ ]:
df_fixed.info()

In [ ]:
df_fixed['Unidades de Processamento'] = pd.to_numeric(df_fixed['Unidades de Processamento'])

In [ ]:
# Removendo valores errados
count = 0
indices = []
for linha in range(len(df_fixed)):
    if df_fixed['Unidades de Processamento'].iloc[linha] > 20000:
        count += 1
        indices.append(linha)
print(count)

In [ ]:
df_fixed = df_fixed.drop(indices)

In [ ]:
df_fixed

In [ ]:
df_fixed['Interface da Mem?ria (Bits)'] = df_fixed['Interface da Mem?ria (Bits)'].astype(str)

In [ ]:
print(df_fixed['Marca'].unique())
print(df_fixed['GPU'].unique())
print(df_fixed['Tipo da Mem?ria'].unique())
print(df_fixed['Interface da Mem?ria (Bits)'].unique())

In [ ]:
import numpy as np
import seaborn as sns
import re
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder 
from sklearn.preprocessing import OrdinalEncoder 
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse=False, categories='auto', dtype='int8')
dataframeenc = pd.DataFrame(encoder.fit_transform(df_fixed[['Marca','GPU']]))
columns = encoder.get_feature_names_out(['Marca','GPU'])
dataframeenc.columns = columns
dataframeenc.index = df_fixed.index
df_fixed_onehot = pd.concat([df_fixed, dataframeenc], axis=1)
df_fixed_onehot = df_fixed_onehot.drop(['Marca', 'GPU'], axis=1)
df_fixed_onehot

In [ ]:
print(df_fixed['Tipo da Mem?ria'].unique())
print(df_fixed['Interface da Mem?ria (Bits)'].unique())

In [ ]:
maping = {
    'DDR6': 0, 
    'GDDR5': 1,
    'GDDR6': 2,
    'GDDR6X': 3,
}

df_fixed_onehot['Tipo da Mem?ria'] = df_fixed_onehot['Tipo da Mem?ria'].replace(maping)
df_fixed_onehot

In [ ]:
maping = {
    '64.0': 0,
    '128.0': 1, 
    '192.0': 2,
    '256.0': 3,
    '384.0': 4,
}

df_fixed_onehot['Interface da Mem?ria (Bits)'] = df_fixed_onehot['Interface da Mem?ria (Bits)'].replace(maping)
df_fixed_onehot

In [ ]:
print(df_fixed_onehot['Tipo da Mem?ria'].unique())
print(df_fixed_onehot['Interface da Mem?ria (Bits)'].unique())

In [ ]:
df_fixed.to_csv("dataset_fixed.csv")

In [ ]:
x = (df_fixed_onehot.drop('Pre?o', axis = 1)).to_numpy()
y = (df_fixed_onehot['Pre?o']).to_numpy()

xtreino, xteste, ytreino, yteste = train_test_split(x,y, train_size=0.8)

print(xtreino.shape)
print(ytreino.shape)
print(xteste.shape)
print(yteste.shape)

In [ ]:
Scaler = MinMaxScaler()
columns = xtreino

xtreino = Scaler.fit_transform(xtreino)
xteste = Scaler.transform(xteste)


Scaleroutro = MinMaxScaler()
ytreino = Scaleroutro.fit_transform(ytreino.reshape(-1,1))
yteste = Scaleroutro.transform(yteste.reshape(-1,1))

# display(ytreino)

In [ ]:
model = LinearRegression()

model.fit(xtreino, ytreino)

prediction = model.predict(xteste)

yteste = Scaleroutro.inverse_transform(yteste)

prediction = Scaleroutro.inverse_transform(prediction)

mean_squared_error(yteste, prediction)

print(np.sqrt(mean_squared_error(yteste, prediction)))

In [ ]:
df_fixed_onehot['Pre?o'].mean()